# 21 — SELFormer Embeddings (HUBioDataLab/SELFormer)
RoBERTa pretrained on SELFIES strings — uses a chemically-valid molecular
representation that eliminates invalid SMILES. 768-dim [CLS] embeddings.
Provides orthogonal signal to SMILES-based transformers by encoding
molecular graphs via a notation guaranteed to parse correctly.
Runtime: ~25 min (CPU inference).

In [ ]:
# ── 1. Setup ──────────────────────────────────────────────────────────────────
import sys, warnings
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import torch
import lightgbm as lgb
import selfies
from transformers import AutoTokenizer, AutoModel
from tqdm.auto import tqdm

from pxr.data import load_train, load_test
from pxr.chem import bemis_murcko
from pxr.eval import scaffold_kfold_indices, compute_metrics, rae as rae_fn
from pxr.paths import DATA_PROCESSED, SUBMISSIONS

SEED = 42
N_FOLDS = 5
MODEL_NAME = 'HUBioDataLab/SELFormer'
MAX_LEN = 128
BATCH_SIZE = 32

print(f"torch {torch.__version__}  |  device: cpu")
print(f"selfies {selfies.__version__}")
print(f"Model: {MODEL_NAME}")

In [ ]:
# ── 2. Load data ──────────────────────────────────────────────────────────────
train = load_train()
te    = load_test()

smiles_tr = train['smiles'].tolist()
smiles_te = te['smiles'].tolist()
y_tr      = train['pec50'].values

scaffolds = train['smiles'].map(bemis_murcko).tolist()
splits    = scaffold_kfold_indices(scaffolds, n_splits=N_FOLDS, seed=SEED)

print(f"Train: {len(smiles_tr):,}  |  Test: {len(smiles_te):,}")
print(f"y_tr range: {y_tr.min():.3f} – {y_tr.max():.3f}")

In [ ]:
# ── 3. SELFIES conversion and SELFormer embeddings ────────────────────────────
def smiles_to_selfies(smi: str) -> str:
    try:
        sf = selfies.encoder(smi)
        return sf if sf else ""
    except Exception:
        return ""

CACHE_TR = DATA_PROCESSED / 'selformer_train_emb.npy'
CACHE_TE = DATA_PROCESSED / 'selformer_test_emb.npy'

def get_selformer_emb(smiles_list: list, cache: Path, tokenizer, model,
                      batch_size: int = 32) -> np.ndarray:
    if cache.exists():
        print(f"  Loading cached {cache.name}")
        return np.load(str(cache))
    selfies_list = [smiles_to_selfies(s) for s in smiles_list]
    n_fail = sum(1 for s in selfies_list if not s)
    print(f"  SELFIES conversion: {len(selfies_list) - n_fail}/{len(selfies_list)} succeeded")
    all_emb = []
    model.eval()
    for i in tqdm(range(0, len(selfies_list), batch_size), desc='Encoding'):
        batch = selfies_list[i:i + batch_size]
        # Replace empty with '[C]' (methane) to avoid tokenizer errors
        batch = [s if s else '[C]' for s in batch]
        enc = tokenizer(batch, return_tensors='pt', padding=True,
                        truncation=True, max_length=MAX_LEN)
        with torch.no_grad():
            out = model(**enc)
        cls = out.last_hidden_state[:, 0].cpu().numpy()
        all_emb.append(cls)
    X = np.vstack(all_emb).astype(np.float32)
    np.save(str(cache), X)
    return X

# Load model
print(f"Loading model from HuggingFace: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModel.from_pretrained(MODEL_NAME)
print(f"SELFormer vocab size: {tokenizer.vocab_size}  hidden: {model.config.hidden_size}")

print("Generating train embeddings ...")
X_tr = get_selformer_emb(smiles_tr, CACHE_TR, tokenizer, model)
print(f"  Train: {X_tr.shape}")
print("Generating test embeddings ...")
X_te = get_selformer_emb(smiles_te, CACHE_TE, tokenizer, model)
print(f"  Test:  {X_te.shape}")
print(f"  NaN check — train: {np.isnan(X_tr).sum()}  test: {np.isnan(X_te).sum()}")

In [ ]:
# ── 4. Scaffold 5-fold CV with LGBM OOF ───────────────────────────────────────
LGBM_PARAMS = dict(
    n_estimators=1000, num_leaves=64, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=0.1,
    min_child_samples=10, n_jobs=4, verbose=-1
)

oof = np.full(len(y_tr), np.nan)
fold_metrics = []

for fold_i, (tr_idx, va_idx) in enumerate(splits):
    m = lgb.LGBMRegressor(**LGBM_PARAMS)
    m.fit(X_tr[tr_idx], y_tr[tr_idx])
    oof[va_idx] = m.predict(X_tr[va_idx])
    fold_rae = rae_fn(y_tr[va_idx], oof[va_idx])
    met = compute_metrics(y_tr[va_idx], oof[va_idx])
    met['fold'] = fold_i
    fold_metrics.append(met)
    print(f"  Fold {fold_i + 1}: RAE={fold_rae:.4f}  Spearman={met['Spearman']:.4f}")

oof_rae = rae_fn(y_tr, oof)
cv_df = pd.DataFrame(fold_metrics)
print(f"\nOOF RAE (global): {oof_rae:.4f}")
print(f"Mean fold RAE:    {cv_df['RAE'].mean():.4f} +/- {cv_df['RAE'].std():.4f}")
print()
print("== Comparison ==")
print(f"  LGBM_base (Morgan + RDKit only): ~0.575")
print(f"  Chemprop multitask (nb 03):       0.517")
print(f"  ChemBERTa-zinc-MLM (nb 13):       0.6782")
print(f"  ChemBERTa-PubChem-MTR (nb 14):    0.5993")
print(f"  BERT-SMILES (nb 20):              0.6XXX")
print(f"  SELFormer (this nb):             {oof_rae:.4f}")

np.save(DATA_PROCESSED / 'oof_selformer.npy', oof)

In [ ]:
# ── 5. Full retrain on all train data + predict test ──────────────────────────
final_m = lgb.LGBMRegressor(**LGBM_PARAMS)
final_m.fit(X_tr, y_tr)
te_preds = final_m.predict(X_te)
te_preds = np.clip(te_preds, y_tr.min() - 0.5, y_tr.max() + 0.5)
np.save(DATA_PROCESSED / 'te_selformer.npy', te_preds)
print(f"Test preds: mean={te_preds.mean():.3f}  std={te_preds.std():.3f}")
print(f"Test preds range: {te_preds.min():.3f} – {te_preds.max():.3f}")

In [ ]:
# ── 6. Save submission ────────────────────────────────────────────────────────
sub = pd.DataFrame({
    'Molecule Name': te['name'].values,
    'SMILES':        te['smiles'].values,
    'pEC50':         te_preds
})
assert len(sub) == 513 and sub['pEC50'].notna().all()
out = SUBMISSIONS / '21_selformer.csv'
sub.to_csv(out, index=False)
print(f"Saved: {out}")
print(f"OOF RAE (SELFormer LGBM): {oof_rae:.4f}")
print(sub['pEC50'].describe().round(3))